In [ ]:
"""
Async HTML Page Scraper
Usage:
    python scraper.py panasonic_pages.json
    python scraper.py panasonic_pages.json --output-dir my_pages --concurrency 5
"""

import asyncio
import aiohttp
import json
import logging
import re
import sys
import time
import argparse
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional


# ---------- CONFIG DEFAULTS ----------
DEFAULT_OUTPUT_DIR   = "html_pages"
DEFAULT_CONCURRENCY  = 10
DEFAULT_TIMEOUT      = 30
DEFAULT_RETRIES      = 3
DEFAULT_RETRY_DELAY  = 1.5   # seconds (exponential base)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}


# ---------- LOGGING ----------
def setup_logging(verbose: bool = False) -> logging.Logger:
    level = logging.DEBUG if verbose else logging.INFO
    logging.basicConfig(
        level=level,
        format="%(asctime)s [%(levelname)s] %(message)s",
        datefmt="%H:%M:%S",
    )
    return logging.getLogger("scraper")


# ---------- DATA MODELS ----------
@dataclass
class Product:
    product_name: str
    product_url: str
    reference: str = ""
    brand: str = ""
    price: str = ""
    availability: str = ""

    @classmethod
    def from_dict(cls, data: dict) -> "Product":
        required = {"product_name", "product_url"}
        missing = required - data.keys()
        if missing:
            raise ValueError(f"Product missing required fields: {missing} → {data}")
        return cls(**{k: data.get(k, "") for k in cls.__dataclass_fields__})


@dataclass
class Stats:
    total: int = 0
    success: int = 0
    failed: int = 0
    skipped: int = 0
    start_time: float = field(default_factory=time.time)

    def elapsed(self) -> float:
        return time.time() - self.start_time

    def summary(self) -> str:
        return (
            f"\n{'─'*45}\n"
            f"  Total   : {self.total}\n"
            f"  ✅ Saved  : {self.success}\n"
            f"  ⏭  Skipped: {self.skipped}\n"
            f"  ❌ Failed : {self.failed}\n"
            f"  Time    : {self.elapsed():.1f}s\n"
            f"{'─'*45}"
        )


# ---------- HELPERS ----------
def safe_filename(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return text.strip("_")[:80]  # cap length


def build_filename(product: Product) -> str:
    parts = [safe_filename(product.product_name)]
    if product.reference:
        parts.append(safe_filename(product.reference))
    return "_".join(parts) + ".html"


def load_products(json_path: str) -> list[Product]:
    path = Path(json_path)
    if not path.exists():
        raise FileNotFoundError(f"Input file not found: {json_path}")
    if path.suffix.lower() != ".json":
        raise ValueError(f"Expected a .json file, got: {json_path}")

    with open(path, encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        raise ValueError("JSON root must be an array of product objects.")

    products, errors = [], []
    for i, item in enumerate(data):
        try:
            products.append(Product.from_dict(item))
        except (ValueError, TypeError) as e:
            errors.append(f"  Item #{i}: {e}")

    if errors:
        logging.getLogger("scraper").warning(
            "Skipped %d invalid item(s):\n%s", len(errors), "\n".join(errors)
        )

    return products


# ---------- FETCH ----------
async def fetch_html(
    session: aiohttp.ClientSession,
    url: str,
    retries: int,
    logger: logging.Logger,
) -> Optional[str]:
    for attempt in range(1, retries + 1):
        try:
            async with session.get(url) as response:
                if response.status == 200:
                    return await response.text()
                if response.status in (403, 404, 410):
                    # No point retrying permanent errors
                    logger.warning("HTTP %s (no retry) → %s", response.status, url)
                    return None
                logger.warning(
                    "HTTP %s (attempt %d/%d) → %s",
                    response.status, attempt, retries, url,
                )
        except asyncio.TimeoutError:
            logger.warning("Timeout (attempt %d/%d) → %s", attempt, retries, url)
        except aiohttp.ClientConnectorError as e:
            logger.warning("Connection error (attempt %d/%d) → %s: %s", attempt, retries, url, e)
        except Exception as e:
            logger.warning("Unexpected error (attempt %d/%d) → %s: %s", attempt, retries, url, e)

        if attempt < retries:
            delay = DEFAULT_RETRY_DELAY * (2 ** (attempt - 1))  # exponential back-off
            await asyncio.sleep(delay)

    return None


# ---------- WORKER ----------
async def process_product(
    session: aiohttp.ClientSession,
    semaphore: asyncio.Semaphore,
    product: Product,
    output_dir: Path,
    retries: int,
    stats: Stats,
    logger: logging.Logger,
    skip_existing: bool,
) -> None:
    filename = build_filename(product)
    file_path = output_dir / filename

    # Optional: skip already-downloaded files
    if skip_existing and file_path.exists():
        logger.info("⏭  Skip (exists): %s", filename)
        stats.skipped += 1
        return

    async with semaphore:
        html = await fetch_html(session, product.product_url, retries, logger)

    if html is None:
        logger.error("❌ Failed: %s → %s", product.product_name, product.product_url)
        stats.failed += 1
        return

    file_path.write_text(html, encoding="utf-8")
    logger.info("✅ Saved : %s", filename)
    stats.success += 1


# ---------- MAIN ----------
async def main(args: argparse.Namespace) -> None:
    logger = setup_logging(args.verbose)

    # Load input
    try:
        products = load_products(args.input)
    except (FileNotFoundError, ValueError, json.JSONDecodeError) as e:
        logger.error("Cannot load input: %s", e)
        sys.exit(1)

    if not products:
        logger.warning("No valid products found. Exiting.")
        sys.exit(0)

    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    stats = Stats(total=len(products))
    logger.info("Starting scrape: %d products → %s", stats.total, output_dir)

    timeout   = aiohttp.ClientTimeout(total=args.timeout)
    connector = aiohttp.TCPConnector(limit=args.concurrency, ssl=False)
    semaphore = asyncio.Semaphore(args.concurrency)

    async with aiohttp.ClientSession(
        connector=connector, timeout=timeout, headers=HEADERS
    ) as session:
        tasks = [
            process_product(
                session, semaphore, product,
                output_dir, args.retries, stats, logger,
                skip_existing=args.skip_existing,
            )
            for product in products
        ]
        await asyncio.gather(*tasks)

    print(stats.summary())


# ---------- CLI ----------
def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Async HTML scraper — reads products from a JSON file.",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    parser.add_argument("input", help="Path to input JSON file (e.g. panasonic_pages.json)")
    parser.add_argument("-o", "--output-dir",  default=DEFAULT_OUTPUT_DIR, help="Directory to save HTML files")
    parser.add_argument("-c", "--concurrency", type=int, default=DEFAULT_CONCURRENCY, help="Max parallel requests")
    parser.add_argument("-t", "--timeout",     type=int, default=DEFAULT_TIMEOUT,     help="Request timeout in seconds")
    parser.add_argument("-r", "--retries",     type=int, default=DEFAULT_RETRIES,     help="Retry attempts per URL")
    parser.add_argument("--skip-existing",     action="store_true", help="Skip URLs whose output file already exists")
    parser.add_argument("-v", "--verbose",     action="store_true", help="Enable debug logging")
    return parser.parse_args()


if __name__ == "__main__":
    args = parse_args()
    # Works in both normal Python and Jupyter/Colab environments
    try:
        loop = asyncio.get_running_loop()
        loop.run_until_complete(main(args))
    except RuntimeError:
        asyncio.run(main(args))